In [17]:
%pip install scikit-learn pandas matplotlib seaborn numpy scipy plotly numpy openpyxl
%pip install statsmodels
%pip install torch
%pip install shap
%pip install --upgrade nbformat>=4.2.0
%pip install --upgrade xgboost>=1.7
%pip install --upgrade kaleido
%pip install optuna
%pip install umap-learn


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [18]:
import pandas as pd

CARBON_XLS  = pd.read_excel("carbon_dataset_experimental.xlsx")
MOF_SIM_XLS = pd.read_excel("mof_dataset_simulation.xlsx")
MOF_EXP_XLS = pd.read_excel("mof_dataset_experimental.xlsx")

print(CARBON_XLS.head())
print(MOF_SIM_XLS.head())
print(MOF_EXP_XLS.head())

   PW (V)  SSA (m2/g)  PV (cm3/g)  I (A/g)    Cg (F/g) Electrode Electrolyte  \
0     4.0      1545.0         7.3     50.0  183.950617   3DG1000     EMIMBF4   
1     4.0      1545.0         7.3     20.0  201.851852   3DG1000     EMIMBF4   
2     4.0      1545.0         7.3     10.0  208.024691   3DG1000     EMIMBF4   
3     4.0      1545.0         7.3      5.0  214.814815   3DG1000     EMIMBF4   
4     4.0      1545.0         7.3      3.0  220.987654   3DG1000     EMIMBF4   

   Density (g/cm^3)  Cv (F/cm^3)  T(K)  
0             0.041     7.541975   298  
1             0.041     8.275926   298  
2             0.041     8.529012   298  
3             0.041     8.807407   298  
4             0.041     9.060494   298  
        qmof_id                           Electrode  pore dimensionality  \
0  qmof-1a2e603  boydwoo_str_m1_o26_o26_pcu_sym_117                    3   
1  qmof-5674ed3     boydwoo_str_m3_o7_o27_pcu_sym_8                    3   
2  qmof-3cc0d7f   boydwoo_str_m1_o25_o25_pcu_

In [19]:
"""
# ======================================================================
# FEATURE ENGINEERING + LATENT TRAINING
# ======================================================================
"""


import os, re, random, warnings
import numpy as np
import pandas as pd
import joblib

warnings.filterwarnings("ignore")

# ===================== REPRODUCIBILITY ======================
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)

import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# ===================== DEVICE ===============================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("🔌 Device:", device)

# ===================== PATHS ================================
OUTDIR = "models_disentangled"
MODEL_DIR = os.path.join(OUTDIR, "models")
META_DIR  = os.path.join(OUTDIR, "metadata")

os.makedirs(OUTDIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(META_DIR, exist_ok=True)

# ===================== INPUT FILES ==========================
CARBON_XLS  = "carbon_dataset_experimental.xlsx"
MOF_SIM_XLS = "mof_dataset_simulation.xlsx"
MOF_EXP_XLS = "mof_dataset_experimental.xlsx"

# ===================== UTIL ================================
def normalize_col(s: str) -> str:
    """
    Strong normalization for robust matching:
      - lowercase
      - normalize micro symbol
      - remove ALL non-alphanumerics
    """
    s = str(s).lower().replace("μ", "u").replace("µ", "u")
    s = re.sub(r"[^a-z0-9]+", "", s)
    return s

def sanitize(s: str) -> str:
    return re.sub(r"[^A-Za-z0-9]+", "_", str(s)).strip("_")

# ===================== FEATURE ENGINEERING ==================
def build_dataframe():
    for fn in [CARBON_XLS, MOF_SIM_XLS, MOF_EXP_XLS]:
        if not os.path.exists(fn):
            raise FileNotFoundError(f"❌ Missing input file: {fn}")

    carbon  = pd.read_excel(CARBON_XLS)
    mof_sim = pd.read_excel(MOF_SIM_XLS)
    mof_exp = pd.read_excel(MOF_EXP_XLS)

    carbon["domain"]  = "carbon_exp"
    mof_exp["domain"] = "mof_exp"
    mof_sim["domain"] = "mof_sim"

    df = pd.concat([carbon, mof_exp, mof_sim], ignore_index=True)

    # Minimal non-feature columns to drop (Paper-1 safe)
    drop_cols = {
        "domain", "name", "ref", "qmof_id", "domain_tok",
        "t", "t(k)"
    }
    drop_cols_norm = {normalize_col(c) for c in drop_cols}

    # Aggressive leakage guard (remove anything that smells like targets/proxies)
    # NOTE: we match against normalized names (alphanumeric only).
    leak_patterns = [
        r"capacitance",
        r"volumetriccapacitance",
        r"\bcg\b",
        r"\bcv\b",
        r"currentdensity",
        r"\biag\b",
        r"\bie\b",
        # removed raw "fg" and "ag" (too many false positives after normalization)
        r"fcm",         # e.g., F/cm...
        r"energy",      # catches energy density columns if present
        r"power",       # catches power density columns if present
    ]

    feature_cols = []
    for c in df.columns:
        nc = normalize_col(c)

        # robust drop check
        if nc in drop_cols_norm:
            continue

        # leakage check
        if any(re.search(p, nc) for p in leak_patterns):
            continue

        feature_cols.append(c)

    # Coerce numeric (non-numeric become NaN)
    df_feat = df[feature_cols].apply(pd.to_numeric, errors="coerce")

    # Drop all-NaN columns
    keep = ~df_feat.isna().all(axis=0)
    df_feat = df_feat.loc[:, keep]
    feature_cols = df_feat.columns.tolist()

    # Keep rows with at least ONE finite feature
    row_keep = np.isfinite(df_feat.to_numpy(dtype=float)).any(axis=1)
    df = df.loc[row_keep].reset_index(drop=True)
    df_feat = df_feat.loc[row_keep].reset_index(drop=True)

    domains = df["domain"].astype(str).values

    print(f"\n📊 Samples retained: {len(df_feat)}")
    print(f"🧬 Raw features retained: {len(feature_cols)}")
    print("📦 Domains:", dict(pd.Series(domains).value_counts()))

    return df_feat, domains, feature_cols

# ===================== BUILD X_input ========================
from sklearn.preprocessing import StandardScaler

class FiniteMeanImputer:
    """
    Simple, reproducible imputer:
      - stores per-column mean computed on finite values only
      - transforms by replacing non-finite with stored mean
    """
    def __init__(self):
        self.col_means_ = None

    def fit(self, X: np.ndarray):
        X = np.asarray(X, dtype=float)
        means = []
        for j in range(X.shape[1]):
            col = X[:, j]
            finite = np.isfinite(col)
            if finite.any():
                means.append(float(col[finite].mean()))
            else:
                means.append(0.0)  # should not happen after all-NaN drop
        self.col_means_ = np.array(means, dtype=float)
        return self

    def transform(self, X: np.ndarray) -> np.ndarray:
        if self.col_means_ is None:
            raise RuntimeError("FiniteMeanImputer is not fit yet.")
        X = np.asarray(X, dtype=float).copy()
        mask = ~np.isfinite(X)
        X[mask] = np.take(self.col_means_, np.where(mask)[1])
        return X

def build_X_input(df_feat: pd.DataFrame):
    X_raw = df_feat.to_numpy(dtype=float)
    X_mask = np.isfinite(X_raw).astype(np.float32)

    # (1) Finite-only mean impute (no zero-bias)
    imputer = FiniteMeanImputer().fit(X_raw)
    X_imp = imputer.transform(X_raw)

    # (optional sanity; not a metric)
    if not np.isfinite(X_imp).all():
        raise ValueError("❌ Imputed features still contain non-finite values.")

    # (2) Scale on the imputed array
    scaler = StandardScaler().fit(X_imp)
    X_scaled = scaler.transform(X_imp).astype(np.float32)

    # Model input = [scaled | mask]
    X_input = np.concatenate([X_scaled, X_mask], axis=1).astype(np.float32)
    return X_input, scaler, imputer

# ===================== MODELS ===============================
class LatentEncoder(nn.Module):
    def __init__(self, in_dim, hidden=256, latent=128, dropout=0.2):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, hidden)
        self.fc2 = nn.Linear(hidden, latent)
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        h = self.drop(F.relu(self.fc1(x)))
        z = self.drop(F.relu(self.fc2(h)))
        return z

class DomainDiscriminator(nn.Module):
    def __init__(self, latent, n_domains, dropout=0.2):
        super().__init__()
        h = max(8, latent // 2)
        self.net = nn.Sequential(
            nn.Linear(latent, h),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(h, n_domains)
        )

    def forward(self, z):
        return self.net(z)

class GradReverse(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, lambd):
        ctx.lambd = float(lambd)
        return x.view_as(x)

    @staticmethod
    def backward(ctx, g):
        return -ctx.lambd * g, None

def grad_reverse(x, lambd: float):
    return GradReverse.apply(x, lambd)

# ===================== UNSUPERVISED REGULARIZER =============
# VICReg-style variance + covariance regularization (no targets)
def vicreg_var_cov_loss(z, eps=1e-4, var_target=1.0):
    """
    Ensures:
      - variance per latent dim not collapsing (std >= var_target)
      - low covariance between latent dimensions
    Returns a scalar torch Tensor WITH gradients.
    """
    z = z - z.mean(dim=0, keepdim=True)

    # variance (penalize std below target)
    std = torch.sqrt(z.var(dim=0) + eps)
    var_loss = torch.mean(F.relu(var_target - std) ** 2)

    N, D = z.shape
    # minor guard: covariance estimate is noisy for very small batches
    if N < 4:
        return var_loss

    # covariance (penalize off-diagonals)
    cov = (z.T @ z) / (N - 1 + 1e-12)
    off = cov - torch.diag(torch.diag(cov))
    cov_loss = (off ** 2).sum() / (D * D)

    return var_loss + cov_loss

# ===================== INFO NCE =============================
def info_nce(z1, z2, tau=0.2, symmetric=True):
    z1 = F.normalize(z1, dim=1)
    z2 = F.normalize(z2, dim=1)

    logits12 = (z1 @ z2.T) / tau
    labels = torch.arange(z1.size(0), device=z1.device)
    loss12 = F.cross_entropy(logits12, labels)

    if not symmetric:
        return loss12

    logits21 = (z2 @ z1.T) / tau
    loss21 = F.cross_entropy(logits21, labels)
    return 0.5 * (loss12 + loss21)

# ===================== TRAINING =============================
MAX_EPOCH = 600
BATCH = 128
PATIENCE = 40

TAU = 0.2
NOISE_STD = 0.01

# Include baseline (0,0) on purpose; trains via VICReg reg
LAMBDA_GRID = [
    (0.0, 0.0),    # baseline latent (regularized)
    (0.05, 0.0),   # domain alignment only
    (0.10, 0.05),  # domain + contrastive
]

def train_latent(X, domains, lamD, lamN, tag,
                 hidden=256, latent=128, dropout=0.2):

    dom_list = sorted(pd.unique(domains))
    dom_to_int = {d: i for i, d in enumerate(dom_list)}
    y_dom = np.array([dom_to_int[d] for d in domains], dtype=int)

    enc = LatentEncoder(X.shape[1], hidden=hidden, latent=latent, dropout=dropout).to(device)
    disc = DomainDiscriminator(latent, len(dom_list), dropout=dropout).to(device)

    # --- minor update: separate optimizers for stability ---
    opt_enc = torch.optim.AdamW(enc.parameters(), lr=1e-3, weight_decay=1e-4)
    opt_disc = torch.optim.AdamW(disc.parameters(), lr=1e-3, weight_decay=1e-4)

    best = np.inf
    best_epoch = -1
    patience_left = PATIENCE
    best_state = None

    X_t = torch.tensor(X, device=device)
    y_t = torch.tensor(y_dom, device=device)

    rng = np.random.default_rng(SEED + int(1000 * lamD) + int(2000 * lamN))
    epochs_trained = 0

    for epoch in range(MAX_EPOCH):
        enc.train(); disc.train()
        epochs_trained += 1

        perm = torch.tensor(rng.permutation(len(X_t)), device=device, dtype=torch.long)

        loss_epoch = 0.0
        nb = 0

        for i in range(0, len(X_t), BATCH):
            idx = perm[i:i+BATCH]
            xb = X_t[idx]
            yb = y_t[idx]

            # ===================== (1) Disc step =====================
            if lamD > 0:
                with torch.no_grad():
                    z_det = enc(xb)
                dom_logits = disc(z_det.detach())
                loss_disc = F.cross_entropy(dom_logits, yb)

                opt_disc.zero_grad(set_to_none=True)
                loss_disc.backward()
                opt_disc.step()

            # ===================== (2) Enc step ======================
            z = enc(xb)

            # Always have a differentiable objective
            loss = vicreg_var_cov_loss(z)

            # Domain adversarial term (lamD weights this)
            if lamD > 0:
                dom_logits_grl = disc(grad_reverse(z, 1.0))  # fixed GRL strength
                loss_dom = F.cross_entropy(dom_logits_grl, yb)
                loss = loss + (lamD * loss_dom)

            # Contrastive term (only if lamN>0)
            if lamN > 0 and xb.size(0) >= 2:
                xb2 = xb + NOISE_STD * torch.randn_like(xb)
                z2 = enc(xb2)
                loss_nce = info_nce(z, z2, tau=TAU, symmetric=True)
                loss = loss + (lamN * loss_nce)

            opt_enc.zero_grad(set_to_none=True)
            loss.backward()
            opt_enc.step()

            loss_epoch += float(loss.detach().cpu().item())
            nb += 1

        loss_epoch = loss_epoch / max(1, nb)

        if np.isfinite(loss_epoch) and loss_epoch < best - 1e-6:
            best = loss_epoch
            best_epoch = epoch
            patience_left = PATIENCE
            best_state = {k: v.detach().clone() for k, v in enc.state_dict().items()}
        else:
            patience_left -= 1
            if patience_left <= 0:
                break

        if (epoch + 1) % 50 == 0:
            print(f"   epoch {epoch+1:4d} | loss={loss_epoch:.5f}")

    if best_state is not None:
        enc.load_state_dict(best_state)

    ckpt_path = os.path.join(MODEL_DIR, f"latent_encoder_{sanitize(tag)}.pt")
    torch.save({
        "state_dict": enc.state_dict(),
        "in_dim": int(X.shape[1]),
        "hidden": int(hidden),
        "latent": int(latent),
        "dropout": float(dropout),
        "lambda_domain": float(lamD),
        "lambda_nce": float(lamN),
        "seed": int(SEED),
        "domains": dom_list,
        "dom_to_int": dom_to_int,
        "epochs_trained": int(epochs_trained),
        "best_epoch": int(best_epoch),
        "best_loss": float(best),
        "tau": float(TAU),
        "noise_std": float(NOISE_STD),
        "notes": [
            "Adversarial stabilized: disc trained on z.detach(); enc trained with GRL.",
            "VICReg covariance skipped when batch N<4.",
        ],
    }, ckpt_path)

    print(f"✅ Saved: {os.path.basename(ckpt_path)} | best_loss={best:.6f} | best_epoch={best_epoch+1 if best_epoch>=0 else 'NA'}")

# ===================== MAIN ================================
def main():
    df_feat, domains, feature_cols = build_dataframe()
    X_input, scaler, imputer = build_X_input(df_feat)

    # Save pipeline outputs for Phase 1 + Phase C+3
    np.save(os.path.join(OUTDIR, "X_input.npy"), X_input)
    joblib.dump(domains, os.path.join(OUTDIR, "domains.pkl"))
    joblib.dump(feature_cols, os.path.join(OUTDIR, "feature_cols.pkl"))
    joblib.dump(scaler, os.path.join(OUTDIR, "feature_scaler.pkl"))
    joblib.dump(imputer, os.path.join(OUTDIR, "feature_imputer.pkl"))

    # Metadata checksum for SI reproducibility
    dom_list = sorted(pd.unique(domains))
    dom_to_int = {d: i for i, d in enumerate(dom_list)}

    meta = {
        "seed": SEED,
        "n_samples": int(X_input.shape[0]),
        "n_features_raw": int(len(feature_cols)),
        "n_features_input": int(X_input.shape[1]),
        "mask_offset": int(len(feature_cols)),
        "lambda_grid": LAMBDA_GRID,
        "tau": TAU,
        "noise_std": NOISE_STD,
        "max_epoch": MAX_EPOCH,
        "batch": BATCH,
        "patience": PATIENCE,
        "domains": dom_list,
        "dom_to_int": dom_to_int,
        "notes": [
            "Scaler fit on finite-only mean-imputed data (FiniteMeanImputer).",
            "Drop/leak checks performed on normalized (alphanumeric-only) column names.",
            "Domain adversarial weight: loss += lambdaD * CE(disc(GRL(z,1.0)), domain).",
            "Stabilized adversarial: separate optimizers; disc on z.detach() then enc with GRL.",
            "VICReg covariance skipped for batch N<4.",
            "Leak patterns: removed raw 'ag' and 'fg' to reduce false positives.",
        ],
    }
    joblib.dump(meta, os.path.join(META_DIR, "paper1_latent_training_meta.pkl"))

    # Train models
    for lamD, lamN in LAMBDA_GRID:
        tag = f"lambdaD{lamD}_lambdaN{lamN}"
        print("\n🧠 Training:", tag)
        train_latent(X_input, domains, lamD, lamN, tag)

    print("\n✅ Paper-1 pipeline complete")
    print("📁 OUTDIR:", os.path.abspath(OUTDIR))
    print("📁 Models:", os.path.abspath(MODEL_DIR))

if __name__ == "__main__":
    main()


🔌 Device: cpu

📊 Samples retained: 242
🧬 Raw features retained: 12
📦 Domains: {'carbon_exp': np.int64(122), 'mof_sim': np.int64(102), 'mof_exp': np.int64(18)}

🧠 Training: lambdaD0.0_lambdaN0.0
   epoch   50 | loss=0.01196
   epoch  100 | loss=0.01134
✅ Saved: latent_encoder_lambdaD0_0_lambdaN0_0.pt | best_loss=0.010997 | best_epoch=85

🧠 Training: lambdaD0.05_lambdaN0.0
   epoch   50 | loss=0.05960
   epoch  100 | loss=0.05959
✅ Saved: latent_encoder_lambdaD0_05_lambdaN0_0.pt | best_loss=0.056809 | best_epoch=74

🧠 Training: lambdaD0.1_lambdaN0.05
   epoch   50 | loss=0.37902
✅ Saved: latent_encoder_lambdaD0_1_lambdaN0_05.pt | best_loss=0.320473 | best_epoch=41

✅ Paper-1 pipeline complete
📁 OUTDIR: c:\Users\hrnbe\Greenbootcamps\Projects\Carbon&MOF\Paper 2\models_disentangled
📁 Models: c:\Users\hrnbe\Greenbootcamps\Projects\Carbon&MOF\Paper 2\models_disentangled\models


In [27]:
"""
================================================================================
MODULAR PIPELINE
================================================================================
"""

# =============================================================================
# IMPORTS
# =============================================================================
import os, re, glob, random, warnings
import numpy as np
import pandas as pd
import joblib

import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.neighbors import NearestNeighbors
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

pio.templates.default = "plotly_white"
warnings.filterwarnings("ignore")

# =============================================================================
# RENDERER (VS CODE SAFE)
# =============================================================================
try:
    pio.renderers.default = "vscode"
except Exception:
    pio.renderers.default = "browser"

# =============================================================================
# CONFIG
# =============================================================================
SEED = 42

KNN_K = 10           # kNN for density + distance
NN_REPORT_K = 5      # kNN edges for connectivity + transfer edges
EDGE_K = 5           # edges for transfer error (kept = NN_REPORT_K; separate if needed)

MC_SAMPLES = 40      # MC-dropout samples
MC_BATCH = 512

BINS = 10
MIN_PAIRS_PER_BIN = 12
MIN_DOMAIN_PAIR_N = 20

BOOT = 300
ALPHA = 0.10  # 90% CI

# For tolerance-based analyses (optional)
THRESHOLDS = {
    "Cg (F/g)": 70.0,
    "Cv (F/cm^3)": 70.0,
}

TRAIN_DOMAINS_FOR_OOD = ["carbon_exp", "mof_sim", "mof_exp"]  # used in SI Table S2

# =============================================================================
# VISUAL IDENTITY
# =============================================================================
DOMAIN_ORDER = ["carbon_exp", "mof_exp", "mof_sim", "CTF_exp"]

DOMAIN_COLORS = {
    "carbon_exp": "#1f77b4",
    "mof_exp":    "#d62728",
    "mof_sim":    "#2ca02c",
    "CTF_exp":    "#9467bd",
}

DOMAIN_SYMBOLS = {
    "carbon_exp": "circle",
    "mof_exp":    "triangle-up",
    "mof_sim":    "square",
    "CTF_exp":    "diamond",
}

ZONE_COLORS = {
    "safe":    "#2ca02c",
    "caution": "#ff7f0e",
    "unsafe":  "#d62728",
}

def category_orders_domain(domains):
    return {"domain": [d for d in DOMAIN_ORDER if d in set(domains)]}

# =============================================================================
# REPRODUCIBILITY
# =============================================================================
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# =============================================================================
# PATHS
# =============================================================================
OUTDIR = "models_disentangled"
MODEL_DIR = os.path.join(OUTDIR, "models")
SHAP_DIR = os.path.join(OUTDIR, "shap")

SAVE_DIR = os.path.join(OUTDIR, "results")
os.makedirs(SAVE_DIR, exist_ok=True)

X_PATH = os.path.join(OUTDIR, "X_input.npy")
DOMAINS_PATH = os.path.join(OUTDIR, "domains.pkl")

Y_PATH = os.path.join(OUTDIR, "Y_targets.npy")
T_PATH = os.path.join(OUTDIR, "target_cols.pkl")

UNC_PATH = os.path.join(OUTDIR, "predictions_ann_uncertainty.pkl")

SCALER_PATH = os.path.join(OUTDIR, "feature_scaler.pkl")
FEAT_PATH = os.path.join(OUTDIR, "feature_cols.pkl")

EXTRA_DATASETS = {
    "carbon_exp": "extra_carbon_exp.xlsx",
    "CTF_exp":    "extra_CTF_exp.xlsx",
}

# =============================================================================
# HELPERS
# =============================================================================
def sanitize(s: str) -> str:
    return re.sub(r"[^A-Za-z0-9]+", "_", str(s)).strip("_")

def save_plot(fig, stem: str):
    """
    Show + save HTML always + PNG if kaleido exists.

    FIX: Prevent duplicate VSCode inline rendering.
         Do NOT call both fig.show() and pio.show(fig).
    """
    fig.show()
    fig.write_html(stem + ".html")
    try:
        fig.write_image(stem + ".png", scale=3)
    except Exception:
        pass

def domain_pair(a: str, b: str) -> str:
    return f"{a}→{b}"

def binned_edges(x, bins=BINS):
    x = np.asarray(x, float)
    m = np.isfinite(x)
    if m.sum() < 10:
        return None
    lo, hi = np.nanpercentile(x[m], 1), np.nanpercentile(x[m], 99)
    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        lo, hi = float(np.nanmin(x[m])), float(np.nanmax(x[m]))
    return np.linspace(lo, hi, bins + 1)

def binned_q90_with_ci(x, y, bins=BINS, min_per_bin=MIN_PAIRS_PER_BIN, boot=BOOT, alpha=ALPHA, seed=SEED):
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    m = np.isfinite(x) & np.isfinite(y)
    x = x[m]; y = y[m]
    if len(x) < 30:
        return pd.DataFrame()

    edges = binned_edges(x, bins=bins)
    if edges is None:
        return pd.DataFrame()

    rng = np.random.default_rng(seed)
    bin_id = np.digitize(x, edges) - 1
    bin_id = np.clip(bin_id, 0, bins - 1)

    rows = []
    for b in range(bins):
        mb = bin_id == b
        if mb.sum() < min_per_bin:
            continue
        xb = x[mb]
        yb = y[mb]

        q90 = float(np.nanpercentile(yb, 90))

        boots = np.zeros(boot, float)
        n = len(yb)
        for i in range(boot):
            samp = rng.integers(0, n, size=n)
            boots[i] = np.nanpercentile(yb[samp], 90)

        lo = float(np.nanpercentile(boots, 100 * (alpha / 2)))
        hi = float(np.nanpercentile(boots, 100 * (1 - alpha / 2)))

        rows.append({
            "bin": int(b),
            "x_mid": float(np.nanmean(xb)),
            "n": int(mb.sum()),
            "q90": q90,
            "q90_lo": lo,
            "q90_hi": hi,
        })
    return pd.DataFrame(rows).sort_values("x_mid")

def threshold_distance_from_q90(dfq, thr):
    if dfq is None or dfq.empty:
        return np.nan
    m = dfq["q90"].values >= float(thr)
    if not np.any(m):
        return np.nan
    return float(dfq.loc[m, "x_mid"].iloc[0])

def add_safe_shading(fig, x0, x1, y0, y1, label="SAFE"):
    if not (np.isfinite(x0) and np.isfinite(x1) and x1 > x0):
        return
    fig.add_shape(
        type="rect", xref="x", yref="y",
        x0=float(x0), x1=float(x1),
        y0=float(y0), y1=float(y1),
        fillcolor="rgba(0,200,0,0.10)",
        line=dict(width=0),
        layer="below",
    )
    fig.add_annotation(
        x=(float(x0) + float(x1)) / 2, y=float(y1),
        xref="x", yref="y",
        text=label, showarrow=False,
        yanchor="bottom",
        font=dict(size=12),
        bgcolor="rgba(255,255,255,0.6)",
        bordercolor="rgba(0,120,0,0.4)",
        borderwidth=1,
    )

# =============================================================================
# LOAD REQUIRED ARTIFACTS
# =============================================================================
for p in [X_PATH, DOMAINS_PATH]:
    if not os.path.exists(p):
        raise FileNotFoundError(f"Missing required artifact: {p}")

X_train = np.load(X_PATH).astype(np.float32)
domains_train = np.asarray(joblib.load(DOMAINS_PATH)).astype(str)

if len(X_train) != len(domains_train):
    raise RuntimeError("X_input.npy and domains.pkl length mismatch.")

print("📦 Training samples:", len(X_train))
u, c = np.unique(domains_train, return_counts=True)
print("📦 Domain counts:", dict(zip(u.tolist(), c.tolist())))

# Optional targets
HAS_TARGETS = os.path.exists(Y_PATH) and os.path.exists(T_PATH)
Y = None
targets = None
if HAS_TARGETS:
    Y = np.asarray(np.load(Y_PATH), float)
    targets = list(joblib.load(T_PATH))
    if Y.shape[0] != X_train.shape[0]:
        print("⚠️ Y_targets row mismatch -> disabling target analyses.")
        HAS_TARGETS = False
        Y = None
        targets = None
    else:
        print("🧪 Targets available:", targets)
else:
    print("ℹ️ Targets missing -> target-dependent analyses will be skipped.")

# =============================================================================
# LOAD LATENT ENCODER
# =============================================================================
class Encoder(nn.Module):
    def __init__(self, in_dim, hidden, latent, dropout):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, hidden)
        self.fc2 = nn.Linear(hidden, latent)
        self.drop = nn.Dropout(dropout)

    def encode(self, x):
        h = self.drop(F.relu(self.fc1(x)))
        z = self.drop(F.relu(self.fc2(h)))
        return z

def load_latent_encoder():
    cands = sorted(glob.glob(os.path.join(MODEL_DIR, "latent_encoder_*.pt")))
    if not cands:
        raise FileNotFoundError(f"No latent_encoder_*.pt found in {MODEL_DIR}")
    ckpt_path = cands[0]
    ckpt = torch.load(ckpt_path, map_location="cpu")
    sd = ckpt["state_dict"]

    net = Encoder(
        sd["fc1.weight"].shape[1],
        sd["fc1.weight"].shape[0],
        sd["fc2.weight"].shape[0],
        ckpt.get("dropout", 0.2),
    )
    net.load_state_dict(sd, strict=False)
    net.eval()
    return net, os.path.basename(ckpt_path)

encoder, encoder_name = load_latent_encoder()
print("🧠 Using latent encoder:", encoder_name)

@torch.no_grad()
def encode_all_eval(X_np, batch=1024):
    Zs = []
    for i in range(0, len(X_np), batch):
        xb = torch.tensor(X_np[i:i+batch]).float()
        Zs.append(encoder.encode(xb).numpy())
    return np.vstack(Zs)

def mc_dropout_latent_uncertainty(X_np, mc=MC_SAMPLES, batch=MC_BATCH):
    """Epistemic uncertainty proxy = mean std of latent dims over MC-dropout passes."""
    encoder.train()
    Zs = []
    with torch.no_grad():
        for s in range(mc):
            torch.manual_seed(SEED + s)
            chunks = []
            for i in range(0, len(X_np), batch):
                xb = torch.tensor(X_np[i:i+batch]).float()
                chunks.append(encoder.encode(xb).numpy())
            Zs.append(np.vstack(chunks))
    Zs = np.stack(Zs, axis=0)          # [mc, N, k]
    U = Zs.std(axis=0).mean(axis=1)    # [N]
    encoder.eval()
    return U.astype(float)

# =============================================================================
# GEOMETRY + EPISTEMIC UNCERTAINTY
# =============================================================================
Z_train = encode_all_eval(X_train)
print("🧬 Latent shape:", Z_train.shape)

# Radius
centroid = np.nanmean(Z_train, axis=0)
latent_radius = np.sqrt(np.sum((Z_train - centroid) ** 2, axis=1))

# kNN metrics
k_use = int(max(3, min(KNN_K, len(Z_train) - 1)))
nbrs = NearestNeighbors(n_neighbors=k_use + 1).fit(Z_train)
dist_all, idx_all = nbrs.kneighbors(Z_train)
latent_distance = dist_all[:, 1:].mean(axis=1)
latent_density = 1.0 / (latent_distance + 1e-12)

# Epistemic uncertainty (load if present, else MC-dropout)
U_epi = None
if os.path.exists(UNC_PATH):
    try:
        pack = joblib.load(UNC_PATH)
        ann_mc = pack.get("ANN_MC", None)
        if ann_mc is not None and "std_epistemic" in ann_mc:
            std_epi = np.asarray(ann_mc["std_epistemic"], float)  # [N,T]
            if std_epi.ndim == 2 and std_epi.shape[0] == len(X_train):
                U_epi = np.nanmean(std_epi, axis=1)
                print("✅ Loaded std_epistemic from predictions_ann_uncertainty.pkl")
    except Exception as e:
        print("⚠️ Failed to load UNC_PATH; will compute MC-dropout uncertainty:", e)

if U_epi is None:
    print("⚠️ Using on-the-fly MC-dropout latent uncertainty (encoder dropout).")
    U_epi = mc_dropout_latent_uncertainty(X_train)
    np.save(os.path.join(SAVE_DIR, "mc_dropout_latent_uncertainty.npy"), U_epi)

core_df = pd.DataFrame({
    "idx": np.arange(len(Z_train), dtype=int),
    "domain": domains_train,
    "latent_radius": latent_radius,
    "latent_distance": latent_distance,
    "latent_density": latent_density,
    "uncertainty_epistemic": U_epi,
})
core_df.to_csv(os.path.join(SAVE_DIR, "core_latent_metrics.csv"), index=False)

# Fig A1: radius by domain
fig = px.violin(
    core_df,
    x="domain", y="latent_radius",
    color="domain",
    color_discrete_map=DOMAIN_COLORS,
    category_orders=category_orders_domain(core_df["domain"]),
    box=True, points="outliers",
    title="<b>Fig. A1 — Latent Radius by Domain</b>",
)
fig.update_layout(showlegend=False)
save_plot(fig, os.path.join(SAVE_DIR, "figA1_latent_radius_by_domain"))

# Fig A2: density by domain
fig = px.violin(
    core_df,
    x="domain", y="latent_density",
    color="domain",
    color_discrete_map=DOMAIN_COLORS,
    category_orders=category_orders_domain(core_df["domain"]),
    box=True, points="outliers",
    title="<b>Fig. A2 — Latent Density by Domain</b>",
)
fig.update_layout(showlegend=False)
save_plot(fig, os.path.join(SAVE_DIR, "figA2_latent_density_by_domain"))

# Fig A3: density vs radius
fig = px.scatter(
    core_df,
    x="latent_radius", y="latent_density",
    color="domain", symbol="domain",
    color_discrete_map=DOMAIN_COLORS,
    symbol_map=DOMAIN_SYMBOLS,
    category_orders=category_orders_domain(core_df["domain"]),
    opacity=0.75,
    title="<b>Fig. A3 — Latent Density vs Radius (domain-coded)</b>",
)
save_plot(fig, os.path.join(SAVE_DIR, "figA3_density_vs_radius_domain"))

# Fig A4: distance vs epistemic uncertainty
fig = px.scatter(
    core_df,
    x="latent_distance", y="uncertainty_epistemic",
    color="domain", symbol="domain",
    color_discrete_map=DOMAIN_COLORS,
    symbol_map=DOMAIN_SYMBOLS,
    category_orders=category_orders_domain(core_df["domain"]),
    opacity=0.75,
    title="<b>Fig. A4 — Latent Distance vs Epistemic Uncertainty</b>",
)
save_plot(fig, os.path.join(SAVE_DIR, "figA4_distance_vs_uncertainty"))

# Traffic-light thresholds (Q90)
DIST_Q90 = float(np.nanpercentile(latent_distance, 90))
UNC_Q90 = float(np.nanpercentile(U_epi, 90))
DIST_THRESH = DIST_Q90
UNC_THRESH = UNC_Q90

def zone(d, u):
    if d <= DIST_THRESH and u <= UNC_THRESH:
        return "safe"
    if d > DIST_THRESH and u > UNC_THRESH:
        return "unsafe"
    return "caution"

core_df["zone"] = [zone(d, u) for d, u in zip(core_df["latent_distance"], core_df["uncertainty_epistemic"])]

fig = px.scatter(
    core_df,
    x="latent_distance", y="uncertainty_epistemic",
    color="zone",
    color_discrete_map=ZONE_COLORS,
    symbol="domain",
    symbol_map=DOMAIN_SYMBOLS,
    opacity=0.70,
    title="<b>Fig. A5 — Traffic-Light Safety Map</b><br>(distance × epistemic uncertainty)",
)
fig.add_vline(x=DIST_THRESH, line_dash="dash", line_color="black")
fig.add_hline(y=UNC_THRESH, line_dash="dash", line_color="black")
save_plot(fig, os.path.join(SAVE_DIR, "figA5_traffic_light_map"))

# =============================================================================
# DOMAIN MIXING (kNN CONNECTIVITY)
# =============================================================================
k_rep = int(max(2, min(NN_REPORT_K, len(Z_train) - 1)))
nn_rep = NearestNeighbors(n_neighbors=k_rep + 1, metric="euclidean").fit(Z_train)
drep, irep = nn_rep.kneighbors(Z_train)

edge_rows = []
for i in range(len(Z_train)):
    for r in range(1, k_rep + 1):
        j = int(irep[i, r])
        edge_rows.append({
            "query_idx": int(i),
            "neighbor_rank": int(r),
            "neighbor_idx": int(j),
            "query_domain": str(domains_train[i]),
            "neighbor_domain": str(domains_train[j]),
            "domain_pair": domain_pair(str(domains_train[i]), str(domains_train[j])),
            "latent_distance": float(drep[i, r]),
        })

df_edges = pd.DataFrame(edge_rows)
df_edges.to_csv(os.path.join(SAVE_DIR, "latent_knn_neighbors.csv"), index=False)

pair_counts = df_edges.groupby("domain_pair").size().reset_index(name="n_edges")
pair_counts = pair_counts.sort_values("n_edges", ascending=False)
pair_counts.to_csv(os.path.join(SAVE_DIR, "latent_domain_pair_edge_counts.csv"), index=False)

fig = px.bar(
    pair_counts,
    x="domain_pair", y="n_edges",
    title="<b>Fig. B1 — Latent Neighborhood Connectivity by Domain Pair (kNN edges)</b>",
)
fig.update_traces(text=[f"{v:.0f}" for v in pair_counts["n_edges"].values], textposition="outside", cliponaxis=False)
fig.update_layout(margin=dict(t=110, b=140), xaxis_title="<b>Domain pair</b>", yaxis_title="<b>kNN edge count</b>")
save_plot(fig, os.path.join(SAVE_DIR, "figB1_domain_pair_knn_edges"))

# =============================================================================
# TARGET-DEPENDENT RELIABILITY
# =============================================================================
if HAS_TARGETS:
    # Transfer error table
    transfer_rows = []
    for tj, tname in enumerate(targets):
        y = np.asarray(Y[:, tj], float)
        ok = np.isfinite(y)

        for _, e in df_edges.iterrows():
            qi = int(e["query_idx"])
            ni = int(e["neighbor_idx"])
            if ok[qi] and ok[ni]:
                transfer_rows.append({
                    "target": str(tname),
                    "query_idx": qi,
                    "neighbor_idx": ni,
                    "latent_distance": float(e["latent_distance"]),
                    "abs_error": float(abs(y[qi] - y[ni])),
                    "query_domain": str(e["query_domain"]),
                    "neighbor_domain": str(e["neighbor_domain"]),
                    "domain_pair": str(e["domain_pair"]),
                    "cross_domain": bool(e["query_domain"] != e["neighbor_domain"]),
                })

    df_transfer = pd.DataFrame(transfer_rows)
    df_transfer.to_csv(os.path.join(SAVE_DIR, "transfer_error_vs_latent_distance.csv"), index=False)

    pair_n = df_transfer.groupby(["target", "domain_pair"]).size().reset_index(name="n")
    good_pairs = set(pair_n[pair_n["n"] >= MIN_DOMAIN_PAIR_N]["domain_pair"].tolist())

    # Fig C1
    df_plot = df_transfer[df_transfer["domain_pair"].isin(good_pairs)].copy() if good_pairs else df_transfer.copy()
    fig = px.scatter(
        df_plot,
        x="latent_distance", y="abs_error",
        color="domain_pair",
        facet_col="target",
        opacity=0.55,
        title="<b>Fig. C1 — Transfer Error vs Latent Distance (domain-pair colored)</b>",
    )
    fig.update_layout(margin=dict(t=110, b=80))
    save_plot(fig, os.path.join(SAVE_DIR, "figC1_transfer_error_vs_distance_domainpair"))

    # Local uncertainty proxy
    k_unc = int(max(3, min(KNN_K, len(Z_train) - 1)))
    nn_unc = NearestNeighbors(n_neighbors=k_unc + 1, metric="euclidean").fit(Z_train)
    d_unc, i_unc = nn_unc.kneighbors(Z_train)
    neigh_idx = i_unc[:, 1:]
    neigh_dist = d_unc[:, 1:]
    latent_radius_nn = neigh_dist.mean(axis=1)

    unc_rows = []
    for tj, tname in enumerate(targets):
        y = np.asarray(Y[:, tj], float)
        for i in range(len(Z_train)):
            neigh_y = y[neigh_idx[i]]
            m = np.isfinite(neigh_y)
            if m.sum() < max(3, k_unc // 2):
                continue
            unc_rows.append({
                "target": str(tname),
                "idx": int(i),
                "domain": str(domains_train[i]),
                "latent_radius": float(latent_radius_nn[i]),
                "latent_density": float(latent_density[i]),
                "local_std": float(np.nanstd(neigh_y[m])),
            })

    df_unc = pd.DataFrame(unc_rows)
    df_unc.to_csv(os.path.join(SAVE_DIR, "local_uncertainty_proxy.csv"), index=False)

    fig = px.scatter(
        df_unc,
        x="latent_radius", y="local_std",
        color="domain", symbol="domain",
        color_discrete_map=DOMAIN_COLORS,
        symbol_map=DOMAIN_SYMBOLS,
        facet_col="target",
        opacity=0.70,
        category_orders=category_orders_domain(df_unc["domain"]),
        title="<b>Fig. D1 — Local Uncertainty Proxy vs Latent Radius (domain-coded)</b>",
    )
    fig.update_layout(margin=dict(t=110, b=80))
    save_plot(fig, os.path.join(SAVE_DIR, "figD1_uncertainty_vs_radius_domain"))

    # Binned envelopes
    def binned_envelope(df, xcol, ycol, bycols, nbins=BINS):
        out = []
        x = df[xcol].to_numpy(float)
        m0 = np.isfinite(x)
        if m0.sum() < 20:
            return pd.DataFrame()

        lo, hi = np.nanpercentile(x[m0], 1), np.nanpercentile(x[m0], 99)
        if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
            lo, hi = float(np.nanmin(x[m0])), float(np.nanmax(x[m0]))
        bins = np.linspace(lo, hi, nbins + 1)

        df2 = df.copy()
        df2["bin"] = np.digitize(df2[xcol], bins) - 1
        df2["bin"] = df2["bin"].clip(0, nbins - 1)

        for keys, g in df2.groupby(bycols + ["bin"]):
            yy = g[ycol].to_numpy(float)
            xx = g[xcol].to_numpy(float)
            m = np.isfinite(xx) & np.isfinite(yy)
            if m.sum() < MIN_PAIRS_PER_BIN:
                continue
            out.append({
                **{bycols[i]: keys[i] for i in range(len(bycols))},
                "bin": int(keys[-1]),
                "x_mid": float(np.nanmean(xx[m])),
                "n": int(m.sum()),
                "y_med": float(np.nanmedian(yy[m])),
                "y_q25": float(np.nanpercentile(yy[m], 25)),
                "y_q75": float(np.nanpercentile(yy[m], 75)),
                "y_q90": float(np.nanpercentile(yy[m], 90)),
            })
        return pd.DataFrame(out)

    env = binned_envelope(
        df_transfer,
        xcol="latent_distance",
        ycol="abs_error",
        bycols=["target", "domain_pair"],
        nbins=BINS
    )
    env.to_csv(os.path.join(SAVE_DIR, "transfer_error_envelopes_binned.csv"), index=False)

    for t in sorted(env["target"].unique()) if not env.empty else []:
        et = env[env["target"] == t].copy()
        pair_ok = et.groupby("domain_pair")["n"].sum().reset_index()
        good = set(pair_ok[pair_ok["n"] >= MIN_DOMAIN_PAIR_N]["domain_pair"].tolist())
        et = et[et["domain_pair"].isin(good)].copy()
        if et.empty:
            continue

        fig = go.Figure()
        for dp in sorted(et["domain_pair"].unique()):
            g = et[et["domain_pair"] == dp].sort_values("x_mid")

            fig.add_trace(go.Scatter(x=g["x_mid"], y=g["y_q75"], mode="lines",
                                     line=dict(width=0), showlegend=False, hoverinfo="skip"))
            fig.add_trace(go.Scatter(x=g["x_mid"], y=g["y_q25"], mode="lines",
                                     line=dict(width=0), fill="tonexty",
                                     name=f"{dp} IQR", opacity=0.18))
            fig.add_trace(go.Scatter(x=g["x_mid"], y=g["y_med"], mode="lines+markers",
                                     name=f"{dp} median"))
            fig.add_trace(go.Scatter(x=g["x_mid"], y=g["y_q90"], mode="lines",
                                     line=dict(dash="dot"), name=f"{dp} 90%"))

        fig.update_layout(
            title=f"<b>Fig. E1 — Binned Transfer Error Envelope vs Latent Distance — {t}</b>",
            xaxis_title="<b>Latent distance</b>",
            yaxis_title="<b>|Δ target| (neighbor transfer error)</b>",
            legend=dict(orientation="h", y=1.12, x=1.0, xanchor="right"),
            margin=dict(t=130, b=80),
        )
        save_plot(fig, os.path.join(SAVE_DIR, f"figE1_error_envelope_{sanitize(t)}"))

    # Density/error + density/uncertainty
    dens_map = core_df.set_index("idx")["latent_density"].to_dict()
    df_transfer["query_density"] = df_transfer["query_idx"].map(dens_map).astype(float)

    df_plot = df_transfer[df_transfer["domain_pair"].isin(good_pairs)].copy() if good_pairs else df_transfer.copy()
    fig = px.scatter(
        df_plot,
        x="query_density", y="abs_error",
        color="domain_pair",
        facet_col="target",
        opacity=0.55,
        title="<b>Fig. F1 — Transfer Error vs Latent Density (domain-pair colored)</b>",
    )
    fig.update_layout(margin=dict(t=110, b=80))
    save_plot(fig, os.path.join(SAVE_DIR, "figF1_transfer_error_vs_density_domainpair"))

    fig = px.scatter(
        df_unc,
        x="latent_density", y="local_std",
        color="domain", symbol="domain",
        color_discrete_map=DOMAIN_COLORS,
        symbol_map=DOMAIN_SYMBOLS,
        facet_col="target",
        opacity=0.70,
        category_orders=category_orders_domain(df_unc["domain"]),
        title="<b>Fig. F2 — Local Uncertainty Proxy vs Latent Density (domain-coded)</b>",
    )
    fig.update_layout(margin=dict(t=110, b=80))
    save_plot(fig, os.path.join(SAVE_DIR, "figF2_uncertainty_vs_density_domain"))

    # Domain-pair reliability + FIXED annotations
    rel_rows = []
    for t in targets:
        dt = df_transfer[df_transfer["target"] == str(t)].copy()
        if dt.empty:
            continue
        for dp in sorted(dt["domain_pair"].unique()):
            g = dt[dt["domain_pair"] == dp].copy()
            if len(g) < MIN_DOMAIN_PAIR_N:
                continue
            q1 = np.nanpercentile(g["latent_distance"].values, 25)
            q3 = np.nanpercentile(g["latent_distance"].values, 75)
            near = g[g["latent_distance"] <= q1]
            far  = g[g["latent_distance"] >= q3]
            rel_rows.append({
                "target": str(t),
                "domain_pair": dp,
                "n": int(len(g)),
                "median_err_near": float(np.nanmedian(near["abs_error"].values)) if len(near) else np.nan,
                "median_err_far":  float(np.nanmedian(far["abs_error"].values)) if len(far) else np.nan,
                "q1_distance": float(q1),
                "q3_distance": float(q3),
            })

    df_rel = pd.DataFrame(rel_rows).sort_values(["target", "domain_pair"])
    df_rel.to_csv(os.path.join(SAVE_DIR, "domain_pair_reliability_table.csv"), index=False)

    fig = px.bar(
        df_rel,
        x="domain_pair",
        y="median_err_far",
        color="target",
        barmode="group",
        title="<b>Fig. G1 — Domain-Pair Transfer Reliability (far-neighbor median error)</b>",
    )

    # FIX: set text per-trace; do NOT overwrite with a global update_traces(text=...)
    for trace in fig.data:
        target_name = trace.name
        sub = df_rel[df_rel["target"] == target_name]
        trace.text = [f"{v:.2f}" if np.isfinite(v) else "" for v in sub["median_err_far"].values]
        trace.textposition = "outside"
        trace.cliponaxis = False

    fig.update_layout(
        xaxis_title="<b>Domain pair</b>",
        yaxis_title="<b>Median |Δ target| (far neighbors)</b>",
        margin=dict(t=130, b=150),
        legend=dict(orientation="h", y=1.12, x=1.0, xanchor="right"),
    )
    save_plot(fig, os.path.join(SAVE_DIR, "figG1_domainpair_reliability_bar"))

    # ---- SAFETY AUDIT ADD-ONS (unchanged) ----
    active_targets = [t for t in THRESHOLDS.keys() if t in targets]
    if active_targets:
        for t in active_targets:
            thr = float(THRESHOLDS[t])
            dt = df_transfer[df_transfer["target"] == t].copy()
            if dt.empty:
                continue

            dfq = binned_q90_with_ci(dt["latent_distance"].values, dt["abs_error"].values, seed=SEED + 777)
            dfq.to_csv(os.path.join(SAVE_DIR, f"q90_global_{sanitize(t)}.csv"), index=False)

            d_star = threshold_distance_from_q90(dfq, thr)

            fig = go.Figure()
            fig.add_trace(go.Scatter(x=dfq["x_mid"], y=dfq["q90"], mode="lines+markers", name="q90(|error|)"))
            fig.add_trace(go.Scatter(
                x=np.r_[dfq["x_mid"], dfq["x_mid"][::-1]],
                y=np.r_[dfq["q90_hi"], dfq["q90_lo"][::-1]],
                fill="toself", opacity=0.18, line=dict(width=0),
                name=f"{int((1-ALPHA)*100)}% CI", hoverinfo="skip"
            ))
            fig.add_hline(y=thr, line_dash="dash", annotation_text=f"tol={thr:.0f}", annotation_position="top left")
            if np.isfinite(d_star):
                fig.add_vline(x=d_star, line_dash="dash", annotation_text=f"d*={d_star:.3g}",
                              annotation_position="top right")

            if not dfq.empty:
                x_min, x_max = float(dfq["x_mid"].min()), float(dfq["x_mid"].max())
                y_min, y_max = 0.0, float(max(dfq["q90_hi"].max(), thr) * 1.05)
                if np.isfinite(d_star):
                    add_safe_shading(fig, x_min, min(d_star, x_max), y_min, y_max, label="SAFE (q90<tol)")
                fig.update_yaxes(range=[y_min, y_max])

            fig.update_layout(
                title=f"<b>Fig. S1 — Global q90(|error|) vs Latent Distance (with CI)</b><br>{t}",
                xaxis_title="<b>Latent distance</b>",
                yaxis_title="<b>q90(|error|)</b>",
                margin=dict(t=130, b=80),
            )
            save_plot(fig, os.path.join(SAVE_DIR, f"figS1_q90_global_{sanitize(t)}"))

    # calibrated P(safe) + risk-coverage (unchanged)
    primary = "Cg (F/g)" if targets and "Cg (F/g)" in targets else (targets[0] if targets else None)
    if primary is not None and primary in targets:
        j = targets.index(primary)
        y = np.asarray(Y[:, j], float)
        thr = float(THRESHOLDS.get(primary, np.nan))

        y_nn_mean_err = np.full(len(y), np.nan, float)
        for i in range(len(y)):
            neigh = idx_all[i, 1: min(idx_all.shape[1], 1 + 5)]
            vals = []
            if np.isfinite(y[i]):
                for jn in neigh:
                    if np.isfinite(y[jn]):
                        vals.append(abs(y[i] - y[jn]))
            if vals:
                y_nn_mean_err[i] = float(np.mean(vals))

        m = np.isfinite(y_nn_mean_err) & np.isfinite(latent_distance) & np.isfinite(U_epi) & np.isfinite(thr)
        if m.sum() >= 60:
            label_safe = (y_nn_mean_err[m] <= thr).astype(int)

            X_feat = np.vstack([latent_distance[m], U_epi[m]]).T
            X_mean = X_feat.mean(axis=0)
            X_std = X_feat.std(axis=0) + 1e-12
            X_std_feat = (X_feat - X_mean) / X_std

            if len(np.unique(label_safe)) >= 2:
                clf = LogisticRegression(random_state=SEED, max_iter=500)
                clf.fit(X_std_feat, label_safe)
                p_safe = clf.predict_proba(X_std_feat)[:, 1]
                auc = roc_auc_score(label_safe, p_safe)

                df_cal = pd.DataFrame({
                    "idx": core_df.loc[m, "idx"].values,
                    "domain": core_df.loc[m, "domain"].values,
                    "latent_distance": latent_distance[m],
                    "uncertainty_epistemic": U_epi[m],
                    "proxy_error": y_nn_mean_err[m],
                    "label_safe": label_safe,
                    "p_safe": p_safe,
                })
                df_cal.to_csv(os.path.join(SAVE_DIR, f"safe_probability_fit_{sanitize(primary)}.csv"), index=False)

                xd = np.linspace(np.nanpercentile(latent_distance, 1), np.nanpercentile(latent_distance, 99), 60)
                xu = np.linspace(np.nanpercentile(U_epi, 1), np.nanpercentile(U_epi, 99), 60)
                DD, UU = np.meshgrid(xd, xu)
                grid = np.vstack([DD.reshape(-1), UU.reshape(-1)]).T
                grid_std = (grid - X_mean) / X_std
                p_grid = clf.predict_proba(grid_std)[:, 1].reshape(UU.shape)

                fig = go.Figure(go.Heatmap(
                    z=p_grid, x=xd, y=xu,
                    colorbar=dict(title="P(safe)"),
                    hovertemplate="dist=%{x:.3g}<br>unc=%{y:.3g}<br>P(safe)=%{z:.3f}<extra></extra>",
                ))
                fig.add_vline(x=DIST_THRESH, line_dash="dash", line_color="black")
                fig.add_hline(y=UNC_THRESH, line_dash="dash", line_color="black")
                fig.update_layout(
                    title=f"<b>Fig. S2 — Calibrated Safety Probability Map</b><br>Target={primary} | AUC={auc:.3f}",
                    xaxis_title="<b>Latent distance</b>",
                    yaxis_title="<b>Epistemic uncertainty</b>",
                    margin=dict(t=130, b=80),
                )
                save_plot(fig, os.path.join(SAVE_DIR, f"figS2_psafe_heatmap_{sanitize(primary)}"))

                df_r = df_cal.copy()
                df_r["risk"] = 1.0 - df_r["p_safe"]
                df_r = df_r.sort_values("risk", ascending=True).reset_index(drop=True)

                cov = np.linspace(0.1, 1.0, 19)
                rows = []
                for ccc in cov:
                    n = int(max(10, round(ccc * len(df_r))))
                    sub = df_r.iloc[:n]
                    rows.append({
                        "coverage": float(n / len(df_r)),
                        "q90_proxy_error": float(np.nanpercentile(sub["proxy_error"], 90)),
                    })
                df_rc = pd.DataFrame(rows)
                df_rc.to_csv(os.path.join(SAVE_DIR, f"rejection_curve_{sanitize(primary)}.csv"), index=False)

                fig = go.Figure()
                fig.add_trace(go.Scatter(x=df_rc["coverage"], y=df_rc["q90_proxy_error"], mode="lines+markers",
                                         name="q90(error proxy)"))
                fig.add_hline(y=thr, line_dash="dash", annotation_text=f"tol={thr:.0f}", annotation_position="top left")
                fig.update_layout(
                    title=f"<b>Fig. S3 — Risk–Coverage Curve (Rejection Rule)</b><br>Target={primary}",
                    xaxis_title="<b>Coverage (retained fraction)</b>",
                    yaxis_title="<b>q90(error proxy)</b>",
                    margin=dict(t=130, b=80),
                )
                save_plot(fig, os.path.join(SAVE_DIR, f"figS3_rejection_curve_{sanitize(primary)}"))
            else:
                print("ℹ️ Only one class in safety labels — skipping logistic calibration.")
        else:
            print("ℹ️ Not enough labeled samples for calibrated P(safe) + rejection rule.")
else:
    print("\nℹ️ Target-dependent analyses skipped (no Y_targets.npy / target_cols.pkl).")

# =============================================================================
# MODULE H — OPTIONAL SHAP OVERLAYS (if present)
# =============================================================================
def load_shap_payloads():
    if not os.path.isdir(SHAP_DIR):
        return {}
    cands = sorted(glob.glob(os.path.join(SHAP_DIR, "shap_ann_*.joblib")))
    out = {}
    for p in cands:
        try:
            payload = joblib.load(p)
            t = payload.get("target", os.path.basename(p))
            out[str(t)] = payload
        except Exception:
            pass
    return out

shap_payloads = load_shap_payloads()
if shap_payloads:
    shap_target = "Cg (F/g)" if "Cg (F/g)" in shap_payloads else sorted(shap_payloads.keys())[0]
    payload = shap_payloads[shap_target]

    if "explain_idx" in payload and "shap_values" in payload:
        ex_idx = np.asarray(payload["explain_idx"], int).reshape(-1)
        sv = np.asarray(payload["shap_values"])
        if sv.ndim == 3:
            sv = np.squeeze(sv)
        if sv.ndim == 2 and ex_idx.max() < len(core_df):
            shap_mag = np.mean(np.abs(sv), axis=1)

            df_sh = pd.DataFrame({
                "idx": ex_idx,
                "domain": domains_train[ex_idx],
                "latent_distance": latent_distance[ex_idx],
                "uncertainty_epistemic": U_epi[ex_idx],
                "mean_abs_shap": shap_mag,
            })
            df_sh.to_csv(os.path.join(SAVE_DIR, f"shap_latent_alignment_{sanitize(shap_target)}.csv"), index=False)

            fig = px.scatter(
                df_sh,
                x="latent_distance", y="uncertainty_epistemic",
                color="mean_abs_shap",
                hover_data=["domain"],
                title=f"<b>Fig. H1 — SHAP Magnitude in Transfer-Safety Space</b><br>Target={shap_target}",
                opacity=0.80,
            )
            fig.add_vline(x=DIST_THRESH, line_dash="dash", line_color="black")
            fig.add_hline(y=UNC_THRESH, line_dash="dash", line_color="black")
            save_plot(fig, os.path.join(SAVE_DIR, f"figH1_shap_in_safety_space_{sanitize(shap_target)}"))

# =============================================================================
# MODULE I — OPTIONAL EXTRA INFERENCE DATASETS + DEPLOYMENT MAP
# =============================================================================
extra_tables = []
if os.path.exists(SCALER_PATH) and os.path.exists(FEAT_PATH):
    scaler = joblib.load(SCALER_PATH)
    features = list(joblib.load(FEAT_PATH))

    def build_X_extra(df_raw: pd.DataFrame):
        Xr = df_raw.reindex(columns=features).apply(pd.to_numeric, errors="coerce").values
        mask = np.isfinite(Xr).astype(np.float32)
        Xs = scaler.transform(np.nan_to_num(Xr, nan=0.0)).astype(np.float32)
        return np.concatenate([Xs, mask], axis=1).astype(np.float32)

    def infer_extra(domain_name, path):
        df = pd.read_excel(path)
        Xx = build_X_extra(df)
        Zx = encode_all_eval(Xx)

        nnx = NearestNeighbors(n_neighbors=k_use + 1).fit(Z_train)
        dx, _ = nnx.kneighbors(Zx)
        dist_x = dx[:, 1:].mean(axis=1)

        Ux = mc_dropout_latent_uncertainty(Xx)

        out = pd.DataFrame({
            "domain": domain_name,
            "latent_distance": dist_x,
            "uncertainty_epistemic": Ux,
            "source": "extra",
        })
        out.to_csv(os.path.join(SAVE_DIR, f"extra_{sanitize(domain_name)}_latent_metrics.csv"), index=False)

        fig = px.scatter(
            out,
            x="latent_distance", y="uncertainty_epistemic",
            color="domain", symbol="domain",
            color_discrete_map=DOMAIN_COLORS,
            symbol_map=DOMAIN_SYMBOLS,
            category_orders=category_orders_domain(out["domain"]),
            opacity=0.85,
            title=f"<b>Fig. I1 — Extra Inference Diagnostics — {domain_name}</b><br>(distance × uncertainty)",
        )
        fig.add_vline(x=DIST_THRESH, line_dash="dash", line_color="black")
        fig.add_hline(y=UNC_THRESH, line_dash="dash", line_color="black")
        save_plot(fig, os.path.join(SAVE_DIR, f"figI1_extra_{sanitize(domain_name)}_distance_uncertainty"))
        return out

    for dom, fp in EXTRA_DATASETS.items():
        if os.path.exists(fp):
            print(f"➕ Extra dataset detected: {fp} as domain={dom}")
            extra_tables.append(infer_extra(dom, fp))
        else:
            print(f"ℹ️ Extra dataset not found (skipped): {fp}")

train_map = core_df[["latent_distance", "uncertainty_epistemic", "domain"]].copy()
train_map["source"] = "train"
if extra_tables:
    extra_map = pd.concat(extra_tables, ignore_index=True)
    all_map = pd.concat([train_map, extra_map], ignore_index=True)
else:
    all_map = train_map.copy()

fig = px.scatter(
    all_map,
    x="latent_distance", y="uncertainty_epistemic",
    color="domain",
    symbol="source",
    color_discrete_map=DOMAIN_COLORS,
    opacity=0.75,
    title="<b>Fig. I2 — Deployment Map (Training + Extra Domains)</b><br>(distance × uncertainty)",
)
fig.add_vline(x=DIST_THRESH, line_dash="dash", line_color="black")
fig.add_hline(y=UNC_THRESH, line_dash="dash", line_color="black")
save_plot(fig, os.path.join(SAVE_DIR, "figI2_deployment_map_train_plus_extra"))

# =============================================================================
# SI TABLES — S1 SAFE FRACTION, S2 OOD SEPARATION
# =============================================================================
safe_rows = []
for dom in sorted(core_df["domain"].unique()):
    sub = core_df[core_df["domain"] == dom]
    n = len(sub)
    safe_rows.append({
        "domain": dom,
        "n_samples": n,
        "safe_frac": float((sub["zone"] == "safe").mean()),
        "caution_frac": float((sub["zone"] == "caution").mean()),
        "unsafe_frac": float((sub["zone"] == "unsafe").mean()),
    })

df_safe = pd.DataFrame(safe_rows)
df_safe.to_csv(os.path.join(SAVE_DIR, "SI_Table_S1_safe_fraction_by_domain.csv"), index=False)
print("\n📊 SI Table S1 — Safe fraction by domain")
print(df_safe.round(3))

train_mask = core_df["domain"].isin(TRAIN_DOMAINS_FOR_OOD)
mu_d_train = core_df.loc[train_mask, "latent_distance"].mean()
sd_d_train = core_df.loc[train_mask, "latent_distance"].std()
mu_u_train = core_df.loc[train_mask, "uncertainty_epistemic"].mean()
sd_u_train = core_df.loc[train_mask, "uncertainty_epistemic"].std()

ood_rows = []
for dom in sorted(core_df["domain"].unique()):
    sub = core_df[core_df["domain"] == dom]
    if len(sub) < 10:
        continue
    mu_d = sub["latent_distance"].mean()
    mu_u = sub["uncertainty_epistemic"].mean()
    ood_rows.append({
        "domain": dom,
        "mean_latent_distance": float(mu_d),
        "mean_uncertainty_epistemic": float(mu_u),
        "distance_shift_z": float((mu_d - mu_d_train) / (sd_d_train + 1e-12)),
        "uncertainty_shift_z": float((mu_u - mu_u_train) / (sd_u_train + 1e-12)),
        "ood_flag": "OOD" if dom not in TRAIN_DOMAINS_FOR_OOD else "train",
    })

df_ood = pd.DataFrame(ood_rows)
df_ood.to_csv(os.path.join(SAVE_DIR, "SI_Table_S2_OOD_separation_score.csv"), index=False)
print("\n🧪 SI Table S2 — OOD separation score")
print(df_ood.round(2))

print("\n✅ MERGED PAPER 2 PIPELINE COMPLETE")
print("📁 Outputs:", os.path.abspath(SAVE_DIR))
print("📄 Core table:", os.path.join(SAVE_DIR, "core_latent_metrics.csv"))


📦 Training samples: 242
📦 Domain counts: {'carbon_exp': 122, 'mof_exp': 18, 'mof_sim': 102}
🧪 Targets available: ['gravimetric_capacitance_F_per_g', 'volumetric_capacitance_F_per_cm3']
🧠 Using latent encoder: latent_encoder_lambdaD0_05_lambdaN0_0.pt
🧬 Latent shape: (242, 128)
⚠️ Using on-the-fly MC-dropout latent uncertainty (encoder dropout).


ℹ️ Not enough labeled samples for calibrated P(safe) + rejection rule.
➕ Extra dataset detected: extra_carbon_exp.xlsx as domain=carbon_exp


➕ Extra dataset detected: extra_CTF_exp.xlsx as domain=CTF_exp



📊 SI Table S1 — Safe fraction by domain
       domain  n_samples  safe_frac  caution_frac  unsafe_frac
0  carbon_exp        122      0.852         0.123        0.025
1     mof_exp         18      0.333         0.667        0.000
2     mof_sim        102      0.892         0.049        0.059

🧪 SI Table S2 — OOD separation score
       domain  mean_latent_distance  mean_uncertainty_epistemic  \
0  carbon_exp                  0.80                        0.91   
1     mof_exp                  0.37                        1.03   
2     mof_sim                  0.85                        0.95   

   distance_shift_z  uncertainty_shift_z ood_flag  
0              0.02                -0.36    train  
1             -0.91                 1.36    train  
2              0.14                 0.19    train  

✅ MERGED PAPER 2 PIPELINE COMPLETE
📁 Outputs: c:\Users\hrnbe\Greenbootcamps\Projects\Carbon&MOF\Paper 2\models_disentangled\results
📄 Core table: models_disentangled\results\core_latent_metri

In [28]:
# ===================== SI SENSITIVITY CONFIG =====================
SI_K_REP = 10                  # increase neighbors
SI_BINS = 6                    # fewer bins → more points per bin
SI_MIN_PAIRS_PER_BIN = 8       # relaxed stability threshold
SI_MIN_DOMAIN_PAIR_N = 15      # relaxed inclusion threshold

# =============================================================================
# SUPPLEMENTARY INFORMATION — SENSITIVITY ANALYSIS (RELAXED THRESHOLDS)
# =============================================================================
print("\n📎 Running SI sensitivity analysis with relaxed thresholds...")

# --- rebuild kNN edges with higher k ---
si_k_rep = int(max(3, min(SI_K_REP, len(Z_train) - 1)))
nn_si = NearestNeighbors(n_neighbors=si_k_rep + 1, metric="euclidean").fit(Z_train)
d_si, i_si = nn_si.kneighbors(Z_train)

si_edge_rows = []
for i in range(len(Z_train)):
    for r in range(1, si_k_rep + 1):
        j = int(i_si[i, r])
        si_edge_rows.append({
            "query_idx": int(i),
            "neighbor_idx": int(j),
            "query_domain": str(domains_train[i]),
            "neighbor_domain": str(domains_train[j]),
            "domain_pair": domain_pair(str(domains_train[i]), str(domains_train[j])),
            "latent_distance": float(d_si[i, r]),
        })

df_edges_si = pd.DataFrame(si_edge_rows)

# --- build relaxed transfer table ---
si_transfer_rows = []
for tj, tname in enumerate(targets):
    y = np.asarray(Y[:, tj], float)
    ok = np.isfinite(y)

    for _, e in df_edges_si.iterrows():
        qi = int(e["query_idx"])
        ni = int(e["neighbor_idx"])
        if ok[qi] and ok[ni]:
            si_transfer_rows.append({
                "target": str(tname),
                "latent_distance": float(e["latent_distance"]),
                "abs_error": float(abs(y[qi] - y[ni])),
                "domain_pair": str(e["domain_pair"]),
            })

df_transfer_si = pd.DataFrame(si_transfer_rows)
df_transfer_si.to_csv(
    os.path.join(SAVE_DIR, "SI_transfer_error_relaxed.csv"), index=False
)

# --- relaxed envelope function ---
def binned_envelope_relaxed(df, xcol, ycol, bycols):
    out = []
    x = df[xcol].to_numpy(float)
    m0 = np.isfinite(x)
    if m0.sum() < 20:
        return pd.DataFrame()

    lo, hi = np.nanpercentile(x[m0], 1), np.nanpercentile(x[m0], 99)
    bins = np.linspace(lo, hi, SI_BINS + 1)

    df2 = df.copy()
    df2["bin"] = np.digitize(df2[xcol], bins) - 1
    df2["bin"] = df2["bin"].clip(0, SI_BINS - 1)

    for keys, g in df2.groupby(bycols + ["bin"]):
        yy = g[ycol].to_numpy(float)
        xx = g[xcol].to_numpy(float)
        m = np.isfinite(xx) & np.isfinite(yy)
        if m.sum() < SI_MIN_PAIRS_PER_BIN:
            continue
        out.append({
            **{bycols[i]: keys[i] for i in range(len(bycols))},
            "bin": int(keys[-1]),
            "x_mid": float(np.nanmean(xx[m])),
            "n": int(m.sum()),
            "y_med": float(np.nanmedian(yy[m])),
            "y_q25": float(np.nanpercentile(yy[m], 25)),
            "y_q75": float(np.nanpercentile(yy[m], 75)),
        })
    return pd.DataFrame(out)

env_si = binned_envelope_relaxed(
    df_transfer_si,
    xcol="latent_distance",
    ycol="abs_error",
    bycols=["target", "domain_pair"],
)

env_si.to_csv(
    os.path.join(SAVE_DIR, "SI_error_envelopes_relaxed.csv"), index=False
)

# --- plot relaxed envelopes ---
for t in sorted(env_si["target"].unique()):
    et = env_si[env_si["target"] == t].copy()
    pair_ok = et.groupby("domain_pair")["n"].sum().reset_index()
    good = set(pair_ok[pair_ok["n"] >= SI_MIN_DOMAIN_PAIR_N]["domain_pair"])
    et = et[et["domain_pair"].isin(good)]

    if et.empty:
        continue

    fig = go.Figure()
    for dp in sorted(et["domain_pair"].unique()):
        g = et[et["domain_pair"] == dp].sort_values("x_mid")

        fig.add_trace(go.Scatter(
            x=g["x_mid"], y=g["y_q75"],
            mode="lines", line=dict(width=0),
            showlegend=False
        ))
        fig.add_trace(go.Scatter(
            x=g["x_mid"], y=g["y_q25"],
            mode="lines", fill="tonexty",
            name=f"{dp} IQR", opacity=0.18
        ))
        fig.add_trace(go.Scatter(
            x=g["x_mid"], y=g["y_med"],
            mode="lines+markers", name=f"{dp} median"
        ))

    fig.update_layout(
        title=f"<b>Fig. Sx — Relaxed Error Envelopes (Sensitivity)</b><br>{t}",
        xaxis_title="<b>Latent distance</b>",
        yaxis_title="<b>|Δ target|</b>",
        margin=dict(t=130, b=80),
    )
    save_plot(fig, os.path.join(SAVE_DIR, f"figSx_relaxed_envelope_{sanitize(t)}"))

print("✅ SI sensitivity analysis complete.")



📎 Running SI sensitivity analysis with relaxed thresholds...


✅ SI sensitivity analysis complete.
